# StackGAN sur CelebA

**Papier :** Han Zhang et al., *StackGAN: Text to Photo-realistic Image Synthesis with Stacked Generative Adversarial Networks*, ICCV 2017. [arXiv:1612.03242](https://arxiv.org/abs/1612.03242)

**Dataset requis — Add Data sur Kaggle :** `jessicali9530/celeba-dataset`

**Activer GPU T4 : Settings → Accelerator → GPU T4**

In [ ]:
!pip install transformers pytorch-fid -q

import os, random, pickle, math, warnings, re, subprocess
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image
from transformers import BertTokenizer, BertModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ===== CHEMINS =====
IMG_DIR   = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'
ATTR_CSV  = '/kaggle/input/datasets/jessicali9530/celeba-dataset/list_attr_celeba.csv'
PART_CSV  = '/kaggle/input/datasets/jessicali9530/celeba-dataset/list_eval_partition.csv'

OUTPUT_DIR = '/kaggle/working'
SAMPLE_DIR = f'{OUTPUT_DIR}/samples'
CKPT_DIR   = f'{OUTPUT_DIR}/checkpoints'
FID_REAL   = f'{OUTPUT_DIR}/fid_real'
FID_FAKE   = f'{OUTPUT_DIR}/fid_fake'
EMB_CACHE  = f'{OUTPUT_DIR}/bert_embeddings.pkl'
CAP_CACHE  = f'{OUTPUT_DIR}/captions.pkl'
for d in [SAMPLE_DIR, CKPT_DIR, FID_REAL, FID_FAKE]:
    os.makedirs(d, exist_ok=True)

# ===== HYPERPARAMS =====
N_TRAIN    = 15000   # ✓ bon compromis qualité/temps
N_VAL      = 1000    # ✓ ok
BATCH_SIZE = 64      # ↑ était 32 — T4 16GB peut gérer 64, plus stable
Z_DIM      = 100     # ✓
COND_DIM   = 128     # ✓
TEXT_DIM   = 256     # ✓
GF_DIM     = 128     # ↑ était 64 — double la capacité du générateur
DF_DIM     = 64      # ✓ garder D plus léger que G
LR_G       = 0.0002  # ✓
LR_D       = 0.0001  # ✓
BETA1      = 0.5     # ✓
KL_WEIGHT  = 2.0     # ✓
EPOCHS_1   = 150     # ↑ était 120 — un peu plus long pour mieux converger
EPOCHS_2   = 150     # ↑ idem
LOG_EVERY  = 20      # ✓
N_FID      = 500     # ✓
SEED       = 42      # ✓

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device == 'cuda': torch.cuda.manual_seed_all(SEED)
print('Config OK')

In [ ]:
def attrs_to_caption(row):
    def has(a): return row.get(a, -1) == 1
    is_male  = has('Male')
    gender   = 'man' if is_male else 'woman'
    pronoun  = 'He'  if is_male else 'She'
    parts    = []
    face = []
    if has('Oval_Face'):       face.append('oval face')
    if has('High_Cheekbones'): face.append('high cheekbones')
    if has('Chubby'):          face.append('chubby face')
    if has('Double_Chin'):     face.append('double chin')
    if face:
        parts.append(f'The {gender} has {", ".join(face[:-1]) + " and " + face[-1] if len(face)>1 else face[0]}.')
    else:
        parts.append(f'The {gender} has a face.')
    if is_male:
        fh = []
        if has('5_o_Clock_Shadow'): fh.append('5 o clock shadow')
        if has('Goatee'):           fh.append('goatee')
        if has('Mustache'):         fh.append('mustache')
        if has('Sideburns'):        fh.append('sideburns')
        if fh: parts.append(f'He sports a {", ".join(fh)}.')
    if has('Bald'):
        parts.append(f'{pronoun} is bald.')
    else:
        style = 'wavy hair' if has('Wavy_Hair') else 'straight hair' if has('Straight_Hair') else 'hair with bangs' if has('Bangs') else 'hair'
        color = ' which is black in colour' if has('Black_Hair') else ' which is blond in colour' if has('Blond_Hair') else ' which is brown in colour' if has('Brown_Hair') else ' which is gray in colour' if has('Gray_Hair') else ''
        parts.append(f'{pronoun} has {style}{color}.')
    det = []
    if has('Big_Lips'):            det.append('big lips')
    if has('Big_Nose'):            det.append('big nose')
    if has('Pointy_Nose'):         det.append('pointy nose')
    if has('Narrow_Eyes'):         det.append('narrow eyes')
    if has('Arched_Eyebrows'):     det.append('arched eyebrows')
    if has('Bushy_Eyebrows'):      det.append('bushy eyebrows')
    if has('Mouth_Slightly_Open'): det.append('a slightly open mouth')
    if det:
        parts.append(f'{pronoun} has {", ".join(det[:-1]) + " and " + det[-1] if len(det)>1 else det[0]}.')
    ap = []
    if has('Young'):       ap.append('young')
    if has('Attractive'):  ap.append('attractive')
    if has('Smiling'):     ap.append('smiling')
    if has('Rosy_Cheeks'): ap.append('rosy cheeks')
    if has('Pale_Skin'):   ap.append('pale skin')
    if ap: parts.append(f'The {gender} looks {", ".join(ap[:3])}.')
    acc = []
    if has('Eyeglasses'):       acc.append('eyeglasses')
    if has('Wearing_Hat'):      acc.append('a hat')
    if has('Wearing_Earrings'): acc.append('earrings')
    if has('Wearing_Necklace'): acc.append('a necklace')
    if has('Wearing_Necktie'):  acc.append('a necktie')
    if has('Heavy_Makeup'):     acc.append('heavy makeup')
    if has('Wearing_Lipstick'): acc.append('lipstick')
    if acc: parts.append(f'{pronoun} is wearing {", ".join(acc)}.')
    return ' '.join(parts)

print('Chargement CelebA...')
attr_df = pd.read_csv(ATTR_CSV)
part_df = pd.read_csv(PART_CSV)
attr_df = attr_df.rename(columns={attr_df.columns[0]: 'filename'})
part_df = part_df.rename(columns={part_df.columns[0]: 'filename', part_df.columns[1]: 'partition'})
merged  = attr_df.merge(part_df, on='filename')
train_df = merged[merged['partition'] == 0].iloc[:N_TRAIN]
val_df   = merged[merged['partition'] == 1].iloc[:N_VAL]
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

if os.path.exists(CAP_CACHE):
    with open(CAP_CACHE,'rb') as f: captions_dict = pickle.load(f)
    print('Captions chargees depuis cache')
else:
    all_df = pd.concat([train_df, val_df])
    captions_dict = {}
    for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc='Captions'):
        captions_dict[row['filename']] = attrs_to_caption(row.to_dict())
    with open(CAP_CACHE,'wb') as f: pickle.dump(captions_dict, f)
    print('Captions sauvegardees!')

print('Exemple:', list(captions_dict.values())[0])

In [ ]:
print('Chargement BERT...')
tokenizer  = BertTokenizer.from_pretrained('bert-base-uncased')
bert       = BertModel.from_pretrained('bert-base-uncased').to(device)
bert.eval()

@torch.no_grad()
def encode_texts(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        batch  = texts[i:i+bs]
        tokens = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        emb    = bert(**tokens).last_hidden_state[:, 0, :].cpu().numpy()
        out.append(emb)
    return np.concatenate(out, axis=0)

if os.path.exists(EMB_CACHE):
    with open(EMB_CACHE,'rb') as f: emb_dict = pickle.load(f)
    print('Embeddings charges depuis cache')
else:
    print('Encodage BERT (~3 min)...')
    fnames   = list(captions_dict.keys())
    embs     = encode_texts([captions_dict[f] for f in fnames])
    emb_dict = {f: embs[i] for i, f in enumerate(fnames)}
    with open(EMB_CACHE,'wb') as f: pickle.dump(emb_dict, f)
    print('Embeddings sauvegardes!')

train_files = [r['filename'] for _,r in train_df.iterrows() if r['filename'] in emb_dict]
val_files   = [r['filename'] for _,r in val_df.iterrows()   if r['filename'] in emb_dict]
print(f'Train: {len(train_files)} | Val: {len(val_files)}')

In [ ]:
class FaceTextDataset(Dataset):
    def __init__(self, files, img_dir, emb_dict, img_size=64):
        self.files=files; self.img_dir=img_dir; self.emb_dict=emb_dict
        self.tf = T.Compose([T.CenterCrop(178), T.Resize((img_size,img_size)),
                             T.RandomHorizontalFlip(), T.ToTensor(),
                             T.Normalize([0.5]*3,[0.5]*3)])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        fname = self.files[idx]
        img   = self.tf(Image.open(os.path.join(self.img_dir, fname)).convert('RGB'))
        emb   = torch.tensor(self.emb_dict[fname], dtype=torch.float32)
        wfname= self.files[random.randint(0, len(self.files)-1)]
        wrong = self.tf(Image.open(os.path.join(self.img_dir, wfname)).convert('RGB'))
        return img, wrong, emb

ds1_tr = FaceTextDataset(train_files, IMG_DIR, emb_dict, 64)
ds2_tr = FaceTextDataset(train_files, IMG_DIR, emb_dict, 128)
ds1_vl = FaceTextDataset(val_files,   IMG_DIR, emb_dict, 64)
ds2_vl = FaceTextDataset(val_files,   IMG_DIR, emb_dict, 128)
dl1    = DataLoader(ds1_tr, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
dl2    = DataLoader(ds2_tr, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
dl1v   = DataLoader(ds1_vl, BATCH_SIZE, shuffle=False, num_workers=2, drop_last=True)
dl2v   = DataLoader(ds2_vl, BATCH_SIZE, shuffle=False, num_workers=2, drop_last=True)
print(f'Batches/epoch: S1={len(dl1)} S2={len(dl2)}')
imgs,_,_ = next(iter(dl1))
grid = make_grid((imgs[:8].clamp(-1,1)+1)/2, nrow=8)
plt.figure(figsize=(16,3)); plt.imshow(grid.permute(1,2,0).numpy())
plt.title('Images reelles CelebA (64x64)'); plt.axis('off'); plt.show()

In [ ]:
class CondAug(nn.Module):
    def __init__(self, td=768, cd=128):
        super().__init__()
        self.fc = nn.Linear(td, cd*2)
    def forward(self, t):
        x = F.leaky_relu(self.fc(t), 0.2)
        mu, lv = x.chunk(2, dim=1)
        c = mu + (0.5*lv).exp()*torch.randn_like(mu) if self.training else mu
        return c, -0.5*(1+lv-mu.pow(2)-lv.exp()).mean()

def up_block(ci, co):
    return nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                         nn.Conv2d(ci,co,3,1,1,bias=False), nn.BatchNorm2d(co), nn.ReLU(True))

class G1(nn.Module):
    def __init__(self, z=100, cd=128, gf=64):
        super().__init__()
        self.ca=CondAug(768,cd); self.gf=gf
        self.fc=nn.Sequential(nn.Linear(z+cd,gf*8*4*4), nn.BatchNorm1d(gf*8*4*4), nn.ReLU(True))
        self.up=nn.Sequential(up_block(gf*8,gf*4), up_block(gf*4,gf*2),
                               up_block(gf*2,gf),   up_block(gf,gf//2),
                               nn.Conv2d(gf//2,3,3,1,1), nn.Tanh())
    def forward(self, z, t):
        c,kl=self.ca(t); x=self.fc(torch.cat([z,c],1)).view(-1,self.gf*8,4,4)
        return self.up(x), kl

class D1(nn.Module):
    def __init__(self, df=64, td=256):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(3,df,4,2,1,bias=False), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df,df*2,4,2,1,bias=False), nn.BatchNorm2d(df*2), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df*2,df*4,4,2,1,bias=False), nn.BatchNorm2d(df*4), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df*4,df*8,4,2,1,bias=False), nn.BatchNorm2d(df*8), nn.LeakyReLU(0.2,True))
        self.tp=nn.Sequential(nn.Linear(768,td), nn.LeakyReLU(0.2,True))
        self.jt=nn.Sequential(nn.Conv2d(df*8+td,df*8,1,bias=False),
                               nn.BatchNorm2d(df*8), nn.LeakyReLU(0.2,True),
                               nn.Flatten(), nn.Linear(df*8*4*4,1))
    def forward(self, img, t):
        h=self.enc(img); tt=self.tp(t).unsqueeze(2).unsqueeze(3).expand(-1,-1,4,4)
        return self.jt(torch.cat([h,tt],1))

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(ch,ch,3,1,1,bias=False), nn.BatchNorm2d(ch), nn.ReLU(True),
                                nn.Conv2d(ch,ch,3,1,1,bias=False), nn.BatchNorm2d(ch))
    def forward(self, x): return x+self.net(x)

class G2(nn.Module):
    def __init__(self, cd=128, gf=64, nr=4):
        super().__init__()
        self.ca=CondAug(768,cd)
        self.ie=nn.Sequential(nn.Conv2d(3,gf,3,1,1,bias=False), nn.ReLU(True),
                               nn.Conv2d(gf,gf*2,4,2,1,bias=False), nn.BatchNorm2d(gf*2), nn.ReLU(True),
                               nn.Conv2d(gf*2,gf*4,4,2,1,bias=False), nn.BatchNorm2d(gf*4), nn.ReLU(True))
        self.jt=nn.Sequential(nn.Conv2d(gf*4+cd,gf*4,3,1,1,bias=False), nn.BatchNorm2d(gf*4), nn.ReLU(True))
        self.rb=nn.Sequential(*[ResBlock(gf*4) for _ in range(nr)])
        self.up=nn.Sequential(up_block(gf*4,gf*2), up_block(gf*2,gf), up_block(gf,gf//2),
                               nn.Conv2d(gf//2,3,3,1,1), nn.Tanh())
    def forward(self, s1, t):
        c,kl=self.ca(t); h=self.ie(s1)
        ct=c.unsqueeze(2).unsqueeze(3).expand(-1,-1,h.size(2),h.size(3))
        return self.up(self.rb(self.jt(torch.cat([h,ct],1)))), kl

class D2(nn.Module):
    def __init__(self, df=64, td=256):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(3,df,4,2,1,bias=False), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df,df*2,4,2,1,bias=False), nn.BatchNorm2d(df*2), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df*2,df*4,4,2,1,bias=False), nn.BatchNorm2d(df*4), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df*4,df*8,4,2,1,bias=False), nn.BatchNorm2d(df*8), nn.LeakyReLU(0.2,True),
            nn.Conv2d(df*8,df*16,4,2,1,bias=False), nn.BatchNorm2d(df*16), nn.LeakyReLU(0.2,True))
        self.tp=nn.Sequential(nn.Linear(768,td), nn.LeakyReLU(0.2,True))
        self.jt=nn.Sequential(nn.Conv2d(df*16+td,df*16,1,bias=False),
                               nn.BatchNorm2d(df*16), nn.LeakyReLU(0.2,True),
                               nn.Flatten(), nn.Linear(df*16*4*4,1))
    def forward(self, img, t):
        h=self.enc(img); tt=self.tp(t).unsqueeze(2).unsqueeze(3).expand(-1,-1,4,4)
        return self.jt(torch.cat([h,tt],1))

def winit(m):
    cn=m.__class__.__name__
    if 'Conv' in cn: nn.init.normal_(m.weight,0.0,0.02)
    elif 'BatchNorm' in cn: nn.init.normal_(m.weight,1.0,0.02); nn.init.constant_(m.bias,0)

def bce(logits, real):
    lbl = torch.ones_like(logits) if real else torch.zeros_like(logits)
    return F.binary_cross_entropy_with_logits(logits, lbl)

print('Architectures definies ok')

In [ ]:
TEST_CAPS = [
    'The woman has oval face and high cheekbones. She has wavy brown hair. She has big lips with arched eyebrows. The woman looks young and attractive. She is wearing lipstick.',
    'The man has high cheekbones. He has straight black hair. He has big nose and bushy eyebrows. The man looks young and attractive.',
    'The woman has wavy blond hair and arched eyebrows. She has a slightly open mouth. The woman looks young and smiling. She is wearing heavy makeup and lipstick.',
    'The man sports a 5 o clock shadow and mustache. His hair is brown. He has narrow eyes. The man looks attractive.',
    'The woman has oval face. She has straight gray hair. She has big lips and pointy nose. The woman looks attractive. She is wearing earrings.',
    'The man has high cheekbones. He has wavy brown hair. He has bushy eyebrows. The man looks young and attractive.',
    'The woman has straight blond hair with bangs. The woman looks young and smiling and rosy cheeks. She is wearing lipstick.',
    'The man has oval face. He has straight black hair. He has arched eyebrows. The man looks young and attractive. He is wearing eyeglasses.',
]

with torch.no_grad():
    test_emb = torch.tensor(encode_texts(TEST_CAPS), dtype=torch.float32).to(device)
    test_z   = torch.randn(len(TEST_CAPS), Z_DIM, device=device)

def show_samples(g1, g2, epoch, stage, hist, save=True):
    g1.eval()
    with torch.no_grad():
        s1, _ = g1(test_z, test_emb)
        imgs   = (g2.eval() or g2)(s1, test_emb)[0] if (stage==2 and g2) else s1
    imgs = (imgs.clamp(-1,1)+1)/2
    grid = make_grid(imgs, nrow=8, padding=2).permute(1,2,0).cpu().numpy()
    d_l  = hist['d'][-1] if hist['d'] else 0
    g_l  = hist['g'][-1] if hist['g'] else 0
    plt.figure(figsize=(18,3))
    plt.imshow(grid)
    plt.title(f'Stage-{stage} ({"64x64" if stage==1 else "128x128"}) — Epoch {epoch} | D={d_l:.4f} G={g_l:.4f}', fontweight='bold')
    plt.axis('off'); plt.tight_layout()
    if save: plt.savefig(f'{SAMPLE_DIR}/s{stage}_ep{epoch:03d}.png', dpi=120, bbox_inches='tight')
    plt.show(); plt.close()

print('Vecteurs de test prets')

In [ ]:
gen1=G1(Z_DIM,COND_DIM,GF_DIM).to(device).apply(winit)
dis1=D1(DF_DIM,TEXT_DIM).to(device).apply(winit)
optG1=optim.Adam(gen1.parameters(), lr=LR_G, betas=(BETA1,0.999))
optD1=optim.Adam(dis1.parameters(), lr=LR_D, betas=(BETA1,0.999))
hist1={'d':[],'g':[],'dv':[],'gv':[]}

for epoch in range(1, EPOCHS_1+1):
    gen1.train(); dis1.train()
    de, ge = [], []
    for real,wrong,emb in dl1:
        real,wrong,emb = real.to(device),wrong.to(device),emb.to(device)
        z = torch.randn(real.size(0),Z_DIM,device=device)
        optD1.zero_grad()
        with torch.no_grad(): fk,_=gen1(z,emb)
        ld=(bce(dis1(real,emb),True)+bce(dis1(wrong,emb),False)+bce(dis1(fk,emb),False))/3
        ld.backward(); optD1.step()
        for _ in range(2):
            optG1.zero_grad()
            fk,kl=gen1(z,emb)
            lg=bce(dis1(fk,emb),True)+KL_WEIGHT*kl
            lg.backward(); optG1.step()
        de.append(ld.item()); ge.append(lg.item())
    hist1['d'].append(np.mean(de)); hist1['g'].append(np.mean(ge))
    if epoch%LOG_EVERY==0 or epoch==1:
        gen1.eval(); dis1.eval(); dv,gv=[],[]
        with torch.no_grad():
            for r,w,e in dl1v:
                r,w,e=r.to(device),w.to(device),e.to(device)
                z=torch.randn(r.size(0),Z_DIM,device=device)
                fk,kl=gen1(z,e)
                dv.append((bce(dis1(r,e),True)+bce(dis1(fk,e),False)).item()/2)
                gv.append((bce(dis1(fk,e),True)+KL_WEIGHT*kl).item())
        hist1['dv'].append(np.mean(dv)); hist1['gv'].append(np.mean(gv))
        show_samples(gen1,None,epoch,1,hist1)
        torch.save({'G1':gen1.state_dict(),'D1':dis1.state_dict(),'epoch':epoch,'hist':hist1},
                   f'{CKPT_DIR}/s1_ep{epoch:03d}.pt')
        print(f'[S1] Ep{epoch:03d} D={hist1["d"][-1]:.4f} G={hist1["g"][-1]:.4f} Dv={hist1["dv"][-1]:.4f} Gv={hist1["gv"][-1]:.4f}')

torch.save({'G1':gen1.state_dict(),'D1':dis1.state_dict(),'hist':hist1}, f'{CKPT_DIR}/stage1_final.pt')
print('Stage-I termine!')

In [ ]:
gen1.eval()
for p in gen1.parameters(): p.requires_grad=False
gen2=G2(COND_DIM,GF_DIM,nr=4).to(device).apply(winit)
dis2=D2(DF_DIM,TEXT_DIM).to(device).apply(winit)
optG2=optim.Adam(gen2.parameters(), lr=LR_G, betas=(BETA1,0.999))
optD2=optim.Adam(dis2.parameters(), lr=LR_D, betas=(BETA1,0.999))
hist2={'d':[],'g':[],'dv':[],'gv':[]}

for epoch in range(1, EPOCHS_2+1):
    gen2.train(); dis2.train()
    de, ge = [], []
    for real,wrong,emb in dl2:
        real,wrong,emb = real.to(device),wrong.to(device),emb.to(device)
        z = torch.randn(real.size(0),Z_DIM,device=device)
        with torch.no_grad(): s1,_=gen1(z,emb)
        optD2.zero_grad()
        with torch.no_grad(): fk,_=gen2(s1,emb)
        ld=(bce(dis2(real,emb),True)+bce(dis2(wrong,emb),False)+bce(dis2(fk,emb),False))/3
        ld.backward(); optD2.step()
        for _ in range(2):
            optG2.zero_grad()
            fk,kl=gen2(s1,emb)
            lg=bce(dis2(fk,emb),True)+KL_WEIGHT*kl
            lg.backward(); optG2.step()
        de.append(ld.item()); ge.append(lg.item())
    hist2['d'].append(np.mean(de)); hist2['g'].append(np.mean(ge))
    if epoch%LOG_EVERY==0 or epoch==1:
        gen2.eval(); dis2.eval(); dv,gv=[],[]
        with torch.no_grad():
            for r,w,e in dl2v:
                r,w,e=r.to(device),w.to(device),e.to(device)
                z=torch.randn(r.size(0),Z_DIM,device=device)
                s1v,_=gen1(z,e); fk,kl=gen2(s1v,e)
                dv.append((bce(dis2(r,e),True)+bce(dis2(fk,e),False)).item()/2)
                gv.append((bce(dis2(fk,e),True)+KL_WEIGHT*kl).item())
        hist2['dv'].append(np.mean(dv)); hist2['gv'].append(np.mean(gv))
        show_samples(gen1,gen2,epoch,2,hist2)
        torch.save({'G1':gen1.state_dict(),'G2':gen2.state_dict(),'D2':dis2.state_dict(),
                    'epoch':epoch,'hist':hist2}, f'{CKPT_DIR}/s2_ep{epoch:03d}.pt')
        print(f'[S2] Ep{epoch:03d} D={hist2["d"][-1]:.4f} G={hist2["g"][-1]:.4f} Dv={hist2["dv"][-1]:.4f} Gv={hist2["gv"][-1]:.4f}')

torch.save({'G1':gen1.state_dict(),'G2':gen2.state_dict(),'D2':dis2.state_dict(),'hist1':hist1,'hist2':hist2},
           f'{CKPT_DIR}/stackgan_final.pt')
print('Stage-II termine!')

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(18,10))
fig.suptitle('Courbes de perte — StackGAN Text2FaceGAN', fontsize=15, fontweight='bold')
ep1v = list(range(LOG_EVERY, EPOCHS_1+1, LOG_EVERY))
ep2v = list(range(LOG_EVERY, EPOCHS_2+1, LOG_EVERY))

ax=axes[0][0]
ax.plot(hist1['d'],'crimson',   lw=1.5, label='D train')
ax.plot(hist1['g'],'steelblue', lw=1.5, label='G train')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6,label='Nash')
ax.set_title('Stage-I Train (64x64)',fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

ax=axes[0][1]
if hist1['dv']:
    ax.plot(ep1v[:len(hist1['dv'])],hist1['dv'],'o-','crimson',   lw=1.5,label='D val')
    ax.plot(ep1v[:len(hist1['gv'])],hist1['gv'],'s-','steelblue', lw=1.5,label='G val')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6)
ax.set_title('Stage-I Validation',fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

ax=axes[1][0]
ax.plot(hist2['d'],'darkorange',lw=1.5,label='D train')
ax.plot(hist2['g'],'teal',      lw=1.5,label='G train')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6,label='Nash')
ax.set_title('Stage-II Train (128x128)',fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

ax=axes[1][1]
if hist2['dv']:
    ax.plot(ep2v[:len(hist2['dv'])],hist2['dv'],'o-','darkorange',lw=1.5,label='D val')
    ax.plot(ep2v[:len(hist2['gv'])],hist2['gv'],'s-','teal',      lw=1.5,label='G val')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6)
ax.set_title('Stage-II Validation',fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

for a in axes.flat: a.set_xlabel('Epoch'); a.set_ylabel('Loss BCE')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curves.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
gen1.eval(); gen2.eval()
print(f'Preparation {N_FID} images pour FID...')
tf_fid = T.Compose([T.CenterCrop(178), T.Resize((128,128)), T.ToTensor()])
for d in [FID_REAL,FID_FAKE]:
    for f in os.listdir(d): os.remove(os.path.join(d,f))
for i, fname in enumerate(val_files[:N_FID]):
    save_image(tf_fid(Image.open(os.path.join(IMG_DIR,fname)).convert('RGB')), f'{FID_REAL}/{i:05d}.png')
    e = torch.tensor(emb_dict[fname], dtype=torch.float32).unsqueeze(0).to(device)
    z = torch.randn(1,Z_DIM,device=device)
    with torch.no_grad(): s1,_=gen1(z,e); s2,_=gen2(s1,e)
    save_image((s2.clamp(-1,1)+1)/2, f'{FID_FAKE}/{i:05d}.png')
    if (i+1)%100==0: print(f'  {i+1}/{N_FID}')

result = subprocess.run(['python','-m','pytorch_fid',FID_REAL,FID_FAKE,'--device',device],
                        capture_output=True, text=True)
out = result.stdout + result.stderr; print(out)
m = re.search(r'FID:\s*([\d.]+)', out)
FID_SCORE = float(m.group(1)) if m else None
if FID_SCORE:
    print(f'FID Score : {FID_SCORE:.2f}')
    print('Excellent!' if FID_SCORE<50 else 'Bon' if FID_SCORE<100 else 'Moyen' if FID_SCORE<200 else 'A ameliorer')

In [ ]:
def generate(descs, n=8, seed=None):
    if seed: torch.manual_seed(seed)
    gen1.eval(); gen2.eval()
    with torch.no_grad():
        embs=torch.tensor(encode_texts(descs),dtype=torch.float32).to(device).repeat_interleave(n,0)
        z=torch.randn(len(descs)*n,Z_DIM,device=device)
        s1,_=gen1(z,embs); imgs,_=gen2(s1,embs)
    return (imgs.clamp(-1,1)+1)/2

test_descs = [
    'The woman has oval face and high cheekbones. She has wavy brown hair. She has big lips with arched eyebrows. The woman looks young and attractive. She is wearing lipstick.',
    'The man sports a 5 o clock shadow. His hair is black. He has big nose with bushy eyebrows. The man looks young and attractive.',
    'The woman has straight blond hair. She has arched eyebrows and a slightly open mouth. The woman looks young and smiling. She is wearing heavy makeup and lipstick.',
    'The man has high cheekbones. He has wavy brown hair. He has narrow eyes. The man looks young and attractive.',
]
for desc in test_descs:
    imgs = generate([desc], n=8, seed=42)
    grid = make_grid(imgs,nrow=8,padding=2).permute(1,2,0).cpu().numpy()
    plt.figure(figsize=(18,3)); plt.imshow(grid)
    plt.title(f'"{desc[:90]}"',fontsize=8,style='italic')
    plt.axis('off'); plt.tight_layout()
    plt.savefig(f'{SAMPLE_DIR}/gen_{abs(hash(desc[:20]))%9999:04d}.png',dpi=120,bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
desc='The woman has oval face and high cheekbones. She has wavy brown hair. She has big lips. The woman looks young and attractive. She is wearing lipstick.'
with torch.no_grad():
    et=torch.tensor(encode_texts([desc]),dtype=torch.float32).to(device).repeat(8,1)
    zt=torch.randn(8,Z_DIM,device=device)
    s1t,_=gen1(zt,et); s2t,_=gen2(s1t,et)
s1v=(s1t.clamp(-1,1)+1)/2; s2v=(s2t.clamp(-1,1)+1)/2
fig,axes=plt.subplots(2,8,figsize=(20,6))
fig.suptitle('Stage-I (64x64) vs Stage-II (128x128)',fontsize=12,fontweight='bold')
for i in range(8):
    axes[0][i].imshow(s1v[i].permute(1,2,0).cpu().numpy())
    axes[0][i].set_title('64x64',fontsize=8,color='navy'); axes[0][i].axis('off')
    axes[1][i].imshow(s2v[i].permute(1,2,0).cpu().numpy())
    axes[1][i].set_title('128x128',fontsize=8,color='darkgreen'); axes[1][i].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_s1_s2.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
torch.manual_seed(0)
imgs_mc=generate(['The woman has oval face. She has wavy brown hair. The woman looks young and attractive.'], n=16)
grid_mc=make_grid(imgs_mc,nrow=8,padding=2).permute(1,2,0).cpu().numpy()
plt.figure(figsize=(20,6)); plt.imshow(grid_mc)
plt.title('Verification mode collapse — 16 z differents, meme caption\nDiversite = pas de mode collapse',
          fontsize=11,fontweight='bold'); plt.axis('off')
plt.savefig(f'{OUTPUT_DIR}/mode_collapse_check.png',dpi=150,bbox_inches='tight')
plt.show()
var_px=imgs_mc.var(dim=0).mean().item()
print(f'Variance pixel : {var_px:.4f} (>0.01 = bonne diversite)')

In [ ]:
fig=plt.figure(figsize=(20,22))
gs=gridspec.GridSpec(4,2,figure=fig,hspace=0.45,wspace=0.35)
fig.suptitle('Dashboard Final — StackGAN Text2FaceGAN sur CelebA',fontsize=14,fontweight='bold',y=0.98)

ax=fig.add_subplot(gs[0,0])
ax.plot(hist1['d'],'crimson',lw=1.5,label='D train'); ax.plot(hist1['g'],'steelblue',lw=1.5,label='G train')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6,label='Nash'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Pertes Stage-I Train',fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss BCE')

ax=fig.add_subplot(gs[0,1])
if hist1['dv']:
    ax.plot(ep1v[:len(hist1['dv'])],hist1['dv'],'o-','crimson',lw=1.5,label='D val')
    ax.plot(ep1v[:len(hist1['gv'])],hist1['gv'],'s-','steelblue',lw=1.5,label='G val')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Pertes Stage-I Validation',fontweight='bold'); ax.set_xlabel('Epoch')

ax=fig.add_subplot(gs[1,0])
ax.plot(hist2['d'],'darkorange',lw=1.5,label='D train'); ax.plot(hist2['g'],'teal',lw=1.5,label='G train')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6,label='Nash'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Pertes Stage-II Train',fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss BCE')

ax=fig.add_subplot(gs[1,1])
if hist2['dv']:
    ax.plot(ep2v[:len(hist2['dv'])],hist2['dv'],'o-','darkorange',lw=1.5,label='D val')
    ax.plot(ep2v[:len(hist2['gv'])],hist2['gv'],'s-','teal',lw=1.5,label='G val')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.6); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Pertes Stage-II Validation',fontweight='bold'); ax.set_xlabel('Epoch')

ax=fig.add_subplot(gs[2,0])
ax.plot(hist1['d'],'crimson',lw=1.5,ls='-',label='D S1'); ax.plot(hist1['g'],'steelblue',lw=1.5,ls='-',label='G S1')
ax.plot(hist2['d'],'darkorange',lw=1.5,ls='--',label='D S2'); ax.plot(hist2['g'],'teal',lw=1.5,ls='--',label='G S2')
ax.axhline(math.log(2),color='green',ls=':',alpha=0.6); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Comparaison S1 vs S2',fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')

ax=fig.add_subplot(gs[2,1])
metrics={'D S1':hist1['d'][-1],'G S1':hist1['g'][-1],'D S2':hist2['d'][-1],'G S2':hist2['g'][-1]}
cols=['crimson','steelblue','darkorange','teal']
bars=ax.bar(metrics.keys(),metrics.values(),color=cols,alpha=0.8,edgecolor='black')
ax.axhline(math.log(2),color='green',ls='--',alpha=0.7,label=f'Nash={math.log(2):.3f}'); ax.legend()
for b,v in zip(bars,metrics.values()): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.005,f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
ax.set_title('Pertes finales par stage',fontweight='bold')

ax=fig.add_subplot(gs[3,:])
with torch.no_grad(): sd,_=gen1(test_z,test_emb); sd2,_=gen2(sd,test_emb)
gd=make_grid((sd2.clamp(-1,1)+1)/2,nrow=8,padding=3).permute(1,2,0).cpu().numpy()
ax.imshow(gd); ax.set_title('Visages generes Stage-II (128x128) — 8 descriptions de test',fontweight='bold'); ax.axis('off')

plt.savefig(f'{OUTPUT_DIR}/dashboard_final.png',dpi=150,bbox_inches='tight')
plt.show()
print('Dashboard sauvegarde!')

print('\n=== RESUME FINAL ===')
print(f'Train: {len(train_files)} | Val: {len(val_files)}')
print(f'S1 final -> D={hist1["d"][-1]:.4f} G={hist1["g"][-1]:.4f}')
print(f'S2 final -> D={hist2["d"][-1]:.4f} G={hist2["g"][-1]:.4f}')
print(f'Nash cible : {math.log(2):.4f}')
if FID_SCORE: print(f'FID Score  : {FID_SCORE:.2f}')
print(f'Mode collapse variance : {var_px:.4f}')